In [1]:
import pyspark

import os
import sys
from pyspark.sql import SparkSession

# Dynamically point PySpark to YOUR current active Python environment
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Network binding fix for local execution
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.driver.host=127.0.0.1 --conf spark.driver.bindAddress=127.0.0.1 pyspark-shell"

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("SQLprac") \
    .master("local[2]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

In [2]:
df1 = spark.read.csv(
    "products_dataset.csv",
    header=True,
    inferSchema=True
)

df1.show()
df1.printSchema()


+----------+----------------+-----------+-------+-----+--------------+
|product_id|    product_name|   category|  brand|price|stock_quantity|
+----------+----------------+-----------+-------+-----+--------------+
|         1| Popular Product|     Sports|   Sony|43853|           199|
|         2|Business Product|Electronics| Adidas|42937|           184|
|         3| Capital Product|Electronics|   Sony|32625|            64|
|         4|  Listen Product|  Furniture|  Apple|42153|           245|
|         5|    Door Product|    Fashion|   Sony|12043|           277|
|         6|  Former Product|    Grocery| Adidas|35769|           257|
|         7|   Place Product|  Furniture|   Sony|48408|           147|
|         8|   Mouth Product|    Grocery|Samsung| 6179|           152|
|         9|    Cold Product|  Furniture|Samsung|49683|           247|
|        10|  Before Product|     Sports| Adidas|44290|           204|
|        11|   Trade Product|    Grocery|   Nike|32894|           176|
|     

In [3]:
df2 = spark.read.csv(
    "orders_dataset.csv",
    header=True,
    inferSchema=True
)

df2.show()
df2.printSchema()

+--------+-----------+----------+----------+----------+--------+----------+-------------------+------------+------------+------------------+
|order_id|customer_id|product_id|order_date| ship_date|quantity|unit_price|discount_percentage|payment_mode|order_status|      sales_amount|
+--------+-----------+----------+----------+----------+--------+----------+-------------------+------------+------------+------------------+
|       1|         45|        27|2025-06-16|2025-06-22|       4|      6479|                 15|  Debit Card|   Delivered|           22028.6|
|       2|         64|        19|2026-02-14|2026-02-21|       3|     27477|                 22|         UPI|     Shipped|          64296.18|
|       3|         15|        50|2026-03-23|2026-03-30|       3|     17041|                 10|         UPI|     Shipped|46010.700000000004|
|       4|         47|        40|2025-10-08|2025-10-16|       1|     12793|                 16| Net Banking|   Delivered|10746.119999999999|
|       5|   

In [4]:
df3 = spark.read.csv(
    "customers_dataset.csv",
    header=True,
    inferSchema=True
)

df3.show()
df3.printSchema()

+-----------+----------------+------+---+---------+-----------+-----------+---------------+------------+
|customer_id|   customer_name|gender|age|     city|      state|signup_date|membership_type|credit_score|
+-----------+----------------+------+---+---------+-----------+-----------+---------------+------------+
|          1| Zachary Randall|  Male| 22|     Pune|Maharashtra| 2023-09-03|         Silver|         614|
|          2|   Morgan Wilson|  Male| 27|     Pune|  Telangana| 2025-09-26|         Silver|         802|
|          3|      Troy Brown|Female| 23|  Chennai| Tamil Nadu| 2026-03-27|         Silver|         619|
|          4|  Crystal Miller|  Male| 56|Bangalore|  Telangana| 2025-07-29|           Gold|         612|
|          5|   Marisa Miller|Female| 58|   Mumbai| Tamil Nadu| 2025-07-17|         Silver|         857|
|          6|Theresa Ferguson|Female| 42|   Mumbai|  Karnataka| 2025-01-28|         Silver|         890|
|          7|  Yvonne Carroll|Female| 27|  Chennai|    

In [5]:
df1.createOrReplaceTempView("products")
df2.createOrReplaceTempView("orders")
df3.createOrReplaceTempView("customers")

In [34]:
spark.sql("""SELECT product_id, product_name, category, price, stock_quantity 
FROM products 
WHERE price > 5000 AND stock_quantity < 20""").show()

+----------+---------------+--------+-----+--------------+
|product_id|   product_name|category|price|stock_quantity|
+----------+---------------+--------+-----+--------------+
|        45|Message Product|  Sports|10758|            11|
+----------+---------------+--------+-----+--------------+



In [7]:
spark.sql("""SELECT customer_id, SUM(sales_amount) AS total_sales FROM orders GROUP BY customer_id""").show()

+-----------+------------------+
|customer_id|       total_sales|
+-----------+------------------+
|         31|          182534.2|
|         85|485519.66000000003|
|         65|         417792.07|
|         53|         222359.14|
|         78| 494396.5199999999|
|        108|         182294.28|
|         34|         221218.12|
|        101|         345966.48|
|        115|         145863.62|
|         81|         117330.44|
|         28|          257819.9|
|         76|         314220.92|
|         26|          39616.41|
|         27|           49635.0|
|         44|223141.33999999997|
|        103|         401280.61|
|         12|         123133.35|
|         91|         143649.16|
|         22|         297891.89|
|         93|         845556.64|
+-----------+------------------+
only showing top 20 rows



In [8]:
spark.sql("""SELECT product_id, SUM(quantity) AS total_quantity FROM orders GROUP BY product_id""").show()

+----------+--------------+
|product_id|total_quantity|
+----------+--------------+
|        31|            25|
|        53|            16|
|        34|            25|
|        28|            32|
|        27|            38|
|        26|            10|
|        44|            41|
|        12|            13|
|        22|            25|
|        47|            28|
|         1|            23|
|        52|            18|
|        13|            21|
|        16|            23|
|         6|            21|
|         3|            32|
|        40|            22|
|        20|            33|
|        57|            19|
|        54|            35|
+----------+--------------+
only showing top 20 rows



In [9]:
spark.sql("""SELECT category, ROUND(AVG(price), 2) AS average_price FROM products GROUP BY category""").show()

+-----------+-------------+
|   category|average_price|
+-----------+-------------+
|    Fashion|      17298.6|
|     Sports|     27627.75|
|    Grocery|      26990.0|
|Electronics|      28045.0|
|  Furniture|     30423.18|
+-----------+-------------+



In [ ]:
#cust purchase amt greather than 100,000
spark.sql("""SELECT  customer_id, SUM(sales_amount) as total_purchase 
FROM orders GROUP BY customer_id HAVING SUM(sales_amount) > 100000""").show()

+-----------+------------------+
|customer_id|    total_purchase|
+-----------+------------------+
|         31|          182534.2|
|         85|485519.66000000003|
|         65|         417792.07|
|         53|         222359.14|
|         78| 494396.5199999999|
|        108|         182294.28|
|         34|         221218.12|
|        101|         345966.48|
|        115|         145863.62|
|         81|         117330.44|
|         28|          257819.9|
|         76|         314220.92|
|         44|223141.33999999997|
|        103|         401280.61|
|         12|         123133.35|
|         91|         143649.16|
|         22|         297891.89|
|         93|         845556.64|
|        111|         331817.35|
|          1|         156784.36|
+-----------+------------------+
only showing top 20 rows



In [12]:
spark.sql(""" SELECT product_id, product_name FROM products ORDER BY price DESC LIMIT 5""").show()

+----------+--------------+
|product_id|  product_name|
+----------+--------------+
|         9|  Cold Product|
|         7| Place Product|
|        15|Return Product|
|        29|   Mrs Product|
|        31|  Best Product|
+----------+--------------+



In [33]:
spark.sql("""SELECT p.category, SUM(o.sales_amount) as total_sales
from products p JOIN orders o ON p.product_id = o.product_id
GROUP BY p.category""").show()

+-----------+--------------------+
|   category|         total_sales|
+-----------+--------------------+
|    Fashion|          5952276.32|
|     Sports|  3847011.2800000003|
|    Grocery|   7728527.300000001|
|Electronics|   5205559.010000001|
|  Furniture|1.0324294070000004E7|
+-----------+--------------------+



In [14]:
spark.sql("""SELECT c.customer_id,c.customer_name,o.sales_amount 
FROM customers c JOIN orders o 
ON c.customer_id = o.customer_id
ORDER BY o.sales_amount DESC
LIMIT 3""").show()

+-----------+---------------+------------------+
|customer_id|  customer_name|      sales_amount|
+-----------+---------------+------------------+
|         22|Aaron Mcconnell|          219585.6|
|         62|  Mitchell Kent|216510.19999999998|
|         89|   Larry Flores|209850.55000000002|
+-----------+---------------+------------------+



In [31]:
spark.sql("""SELECT c.customer_id,c.customer_name,SUM(o.sales_amount) as total_purchase 
FROM customers c JOIN orders o 
ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name
ORDER BY total_purchase DESC
LIMIT 3""").show()

+-----------+----------------+-----------------+
|customer_id|   customer_name|   total_purchase|
+-----------+----------------+-----------------+
|         93|Shirley Harrison|        845556.64|
|          6|Theresa Ferguson|        720304.25|
|         11|   Darrell Jones|592001.0900000001|
+-----------+----------------+-----------------+



In [15]:
spark.sql("""SELECT c.membership_type, COUNT(o.order_id) AS order_count 
FROM customers c JOIN orders o
ON c.customer_id = o.customer_id
GROUP BY c.membership_type """).show()

+---------------+-----------+
|membership_type|order_count|
+---------------+-----------+
|       Platinum|        162|
|         Silver|        200|
|           Gold|        138|
+---------------+-----------+



In [16]:
spark.sql("""SELECT c.customer_id, c.customer_name FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id 
WHERE o.order_id IS NULL""").show()

+-----------+-------------+
|customer_id|customer_name|
+-----------+-------------+
|         37|   Robin Cobb|
+-----------+-------------+



In [38]:
spark.sql("""select c.city, count(o.order_id) as order_count, sum(o.sales_amount) as total_sales
from customers c left join orders o on c.customer_id = o.customer_id
group by c.city """).show()

+---------+-----------+------------------+
|     city|order_count|       total_sales|
+---------+-----------+------------------+
|Bangalore|         86| 6147224.900000001|
|  Chennai|         92| 6431761.559999999|
|   Mumbai|         85|        5405864.12|
|     Pune|         83| 4735115.739999999|
|    Delhi|         88|6172928.7299999995|
|Hyderabad|         66|4164772.9300000006|
+---------+-----------+------------------+



In [23]:
spark.sql("""select p.product_id, p.product_name
from products p left anti join orders o on p.product_id = o.product_id """).show()
spark.sql("""select p.product_id, p.product_name
from products p left join orders o on p.product_id = o.product_id
where o.order_id is null """).show()

+----------+------------+
|product_id|product_name|
+----------+------------+
+----------+------------+

+----------+------------+
|product_id|product_name|
+----------+------------+
+----------+------------+



In [35]:
spark.sql(""" select 
customer_id, UPPER(customer_name) AS customer_name
from customers
""").show()

+-----------+----------------+
|customer_id|   customer_name|
+-----------+----------------+
|          1| ZACHARY RANDALL|
|          2|   MORGAN WILSON|
|          3|      TROY BROWN|
|          4|  CRYSTAL MILLER|
|          5|   MARISA MILLER|
|          6|THERESA FERGUSON|
|          7|  YVONNE CARROLL|
|          8|   MICHAEL SLOAN|
|          9|   MARK THOMPSON|
|         10|     TARA WALKER|
|         11|   DARRELL JONES|
|         12| ANTHONY DELGADO|
|         13|  WILLIAM BAILEY|
|         14|  LISA LOPEZ DDS|
|         15|  EMILY GALLOWAY|
|         16|      RYAN COHEN|
|         17|     JACOB LOPEZ|
|         18|   JAIME VASQUEZ|
|         19|   HECTOR GORDON|
|         20|    PAUL STEVENS|
+-----------+----------------+
only showing top 20 rows



In [37]:
spark.sql(""" select concat(customer_name, '-', city) as customer_details
from customers
""").show(truncate=False)

+------------------------+
|customer_details        |
+------------------------+
|Zachary Randall-Pune    |
|Morgan Wilson-Pune      |
|Troy Brown-Chennai      |
|Crystal Miller-Bangalore|
|Marisa Miller-Mumbai    |
|Theresa Ferguson-Mumbai |
|Yvonne Carroll-Chennai  |
|Michael Sloan-Mumbai    |
|Mark Thompson-Chennai   |
|Tara Walker-Bangalore   |
|Darrell Jones-Chennai   |
|Anthony Delgado-Pune    |
|William Bailey-Pune     |
|Lisa Lopez DDS-Pune     |
|Emily Galloway-Delhi    |
|Ryan Cohen-Chennai      |
|Jacob Lopez-Chennai     |
|Jaime Vasquez-Pune      |
|Hector Gordon-Mumbai    |
|Paul Stevens-Delhi      |
+------------------------+
only showing top 20 rows



In [39]:
spark.sql(""" 
select c.city,c.customer_id, c.customer_name, SUM(o.sales_amount) as total_sales
from customers c left join orders o on c.customer_id = o.customer_id
group by c.city, c.customer_id, c.customer_name
""").show()

+---------+-----------+------------------+------------------+
|     city|customer_id|     customer_name|       total_sales|
+---------+-----------+------------------+------------------+
|     Pune|         61|  Brandon Hatfield|139609.40000000002|
|    Delhi|         23|      Joel Griffin|           71356.8|
|Hyderabad|         36|      Amy Sullivan|177272.37999999998|
|    Delhi|         87| Ashley Fitzgerald|         207361.45|
|Bangalore|         58|      Alison Ramos| 88695.73999999999|
|   Mumbai|         75|        Michael Li|           90911.2|
|    Delhi|        102|         James Fox|         272220.37|
|Hyderabad|         51|   Charles Stevens|         264017.62|
|   Mumbai|         86|Elizabeth Sheppard|348116.95999999996|
|Bangalore|         89|      Larry Flores|209850.55000000002|
|  Chennai|         42|   Valerie Johnson|         191127.09|
|Bangalore|         46|     Johnny Warren|217500.52000000002|
|  Chennai|        120|          Terry Ho|         211696.32|
|  Chenn

In [42]:
spark.sql(""" 
with customer_sales as (
select c.city,c.customer_id, c.customer_name, SUM(o.sales_amount) as total_sales
from customers c left join orders o on c.customer_id = o.customer_id
group by c.city, c.customer_id, c.customer_name
),

ranked_customers as(
select *, row_number() over (partition by city order by total_sales desc) as rn from customer_sales)

select city, customer_id, customer_name, total_sales from ranked_customers where rn = 1
""").show()

+---------+-----------+----------------+------------------+
|     city|customer_id|   customer_name|       total_sales|
+---------+-----------+----------------+------------------+
|Bangalore|        106|    Shane Miller|          581160.2|
|  Chennai|         11|   Darrell Jones|         592001.09|
|    Delhi|         93|Shirley Harrison|         845556.64|
|Hyderabad|         45|    Jason Nelson|         591651.49|
|   Mumbai|          6|Theresa Ferguson|         720304.25|
|     Pune|         82|   Kelly Jackson|411115.99999999994|
+---------+-----------+----------------+------------------+



In [43]:
spark.sql(""" select order_id, order_date, year(order_date) as year, month(order_date) as month,
day(order_date) as day from orders
""").show()

+--------+----------+----+-----+---+
|order_id|order_date|year|month|day|
+--------+----------+----+-----+---+
|       1|2025-06-16|2025|    6| 16|
|       2|2026-02-14|2026|    2| 14|
|       3|2026-03-23|2026|    3| 23|
|       4|2025-10-08|2025|   10|  8|
|       5|2026-01-23|2026|    1| 23|
|       6|2026-04-23|2026|    4| 23|
|       7|2026-03-21|2026|    3| 21|
|       8|2025-08-20|2025|    8| 20|
|       9|2025-10-23|2025|   10| 23|
|      10|2025-10-07|2025|   10|  7|
|      11|2026-02-20|2026|    2| 20|
|      12|2025-11-24|2025|   11| 24|
|      13|2025-11-26|2025|   11| 26|
|      14|2026-04-19|2026|    4| 19|
|      15|2025-08-11|2025|    8| 11|
|      16|2025-12-23|2025|   12| 23|
|      17|2025-11-26|2025|   11| 26|
|      18|2026-03-22|2026|    3| 22|
|      19|2025-05-17|2025|    5| 17|
|      20|2025-10-28|2025|   10| 28|
+--------+----------+----+-----+---+
only showing top 20 rows



In [44]:
spark.sql(""" select order_id, order_date, ship_date, datediff(ship_date, order_date) as shipping_duration
from orders 
""").show()

+--------+----------+----------+-----------------+
|order_id|order_date| ship_date|shipping_duration|
+--------+----------+----------+-----------------+
|       1|2025-06-16|2025-06-22|                6|
|       2|2026-02-14|2026-02-21|                7|
|       3|2026-03-23|2026-03-30|                7|
|       4|2025-10-08|2025-10-16|                8|
|       5|2026-01-23|2026-01-31|                8|
|       6|2026-04-23|2026-05-02|                9|
|       7|2026-03-21|2026-03-24|                3|
|       8|2025-08-20|2025-08-24|                4|
|       9|2025-10-23|2025-10-26|                3|
|      10|2025-10-07|2025-10-12|                5|
|      11|2026-02-20|2026-03-01|                9|
|      12|2025-11-24|2025-11-28|                4|
|      13|2025-11-26|2025-12-03|                7|
|      14|2026-04-19|2026-04-21|                2|
|      15|2025-08-11|2025-08-14|                3|
|      16|2025-12-23|2025-12-25|                2|
|      17|2025-11-26|2025-11-30